# Crisis Supporter Journey

This notebook builds a stricter session-based view of the volunteer journey starting from the crisis supporter page.


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import (
    build_date_params,
    default_query_window,
    dry_run_bytes,
    get_client,
    load_sql_template,
    run_query,
)

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 92 #@param {type:"integer"}

VOLUNTEER_PAGE = "https://www.lifeline.org.au/get-involved/volunteer" #@param {type:"string"}
CRISIS_SUPPORTER_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter" #@param {type:"string"}
PHONE_SUPPORT_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/phone-crisis-support" #@param {type:"string"}
DIGITAL_SUPPORT_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/digital-support" #@param {type:"string"}
PHONE_FORM_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/phone-crisis-support/phone-crisis-support" #@param {type:"string"}

DIGITAL_OUTBOUND_DOMAIN = "events.teams.microsoft.com" #@param {type:"string"}
PHONE_FORM_ID = "webform-submission-volunteer-form-paragraph-3216-add-form" #@param {type:"string"}

window = default_query_window(days_back=DAYS_BACK)
analysis_start = window.start_date
analysis_end = window.end_date
analysis_window_label = f"{analysis_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d}"

print(f"Project: {config.PROJECT_ID}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Window: {analysis_start} to {analysis_end}")
print(f"Starting cohort page: {CRISIS_SUPPORTER_PAGE}")


In [ ]:
query = load_sql_template(
    "sql/notebooks/ga4_volunteer_crisis_supporter_session_paths.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("volunteer_page", "STRING", VOLUNTEER_PAGE),
    bigquery.ScalarQueryParameter("crisis_supporter_page", "STRING", CRISIS_SUPPORTER_PAGE),
    bigquery.ScalarQueryParameter("phone_support_page", "STRING", PHONE_SUPPORT_PAGE),
    bigquery.ScalarQueryParameter("digital_support_page", "STRING", DIGITAL_SUPPORT_PAGE),
    bigquery.ScalarQueryParameter("phone_form_page", "STRING", PHONE_FORM_PAGE),
    bigquery.ScalarQueryParameter("digital_outbound_domain", "STRING", DIGITAL_OUTBOUND_DOMAIN),
    bigquery.ScalarQueryParameter("phone_form_id", "STRING", PHONE_FORM_ID),
]

estimated_bytes = dry_run_bytes(client, query, params=params)
print(f"Estimated bytes scanned: {estimated_bytes:,}")

sessions = run_query(client, query, params=params)
print(f"Crisis supporter cohort sessions: {len(sessions):,}")

if sessions.empty:
    print("No crisis supporter sessions found in the selected window.")
else:
    sessions.head()


In [ ]:
cohort_sessions = len(sessions)

if cohort_sessions == 0:
    raise ValueError("No sessions returned for the selected window.")

metric_counts = {
    "Volunteer page before crisis page": int(sessions["reached_volunteer_before_crisis"].sum()),
    "Digital support page after crisis": int(sessions["reached_digital_after_crisis"].sum()),
    "Teams signup after digital page": int(sessions["reached_digital_signup_after_digital"].sum()),
    "Phone support page after crisis": int(sessions["reached_phone_support_after_crisis"].sum()),
    "Phone form page after phone support": int(sessions["reached_phone_form_after_support"].sum()),
    "Phone submit after form": int(sessions["reached_phone_submit_after_form"].sum()),
    "Other page after crisis": int(sessions["has_other_page_after_crisis"].sum()),
    "No onward activity": int((sessions["has_post_crisis_activity"] == 0).sum()),
    "Engaged but no tracked branch": int(((sessions["engaged_session"] == 1) & (sessions["onward_outcome"] != "Reached tracked branch")).sum()),
    "Other page only": int((sessions["onward_outcome"] == "Other page only").sum()),
    "Other activity only": int((sessions["onward_outcome"] == "Other activity only").sum()),
    "GA4 bounced sessions": int((sessions["engaged_session"] == 0).sum()),
}

summary = pd.DataFrame(
    [
        ["Crisis supporter cohort sessions", cohort_sessions, "count"],
        ["Reached volunteer page before crisis", metric_counts["Volunteer page before crisis page"], "count"],
        ["Reached digital support after crisis", metric_counts["Digital support page after crisis"], "count"],
        ["Reached Teams signup after digital", metric_counts["Teams signup after digital page"], "count"],
        ["Digital conversion rate from crisis cohort", metric_counts["Teams signup after digital page"] / cohort_sessions, "rate"],
        ["Reached phone support after crisis", metric_counts["Phone support page after crisis"], "count"],
        ["Reached phone form after support", metric_counts["Phone form page after phone support"], "count"],
        ["Reached phone submit after form", metric_counts["Phone submit after form"], "count"],
        ["Phone conversion rate from crisis cohort", metric_counts["Phone submit after form"] / cohort_sessions, "rate"],
        ["Reached any other page after crisis", metric_counts["Other page after crisis"], "count"],
        ["Other page only after crisis", metric_counts["Other page only"], "count"],
        ["Other activity only after crisis", metric_counts["Other activity only"], "count"],
        ["No onward activity after crisis", metric_counts["No onward activity"], "count"],
        ["Engaged but no tracked branch", metric_counts["Engaged but no tracked branch"], "count"],
        ["GA4 bounced sessions", metric_counts["GA4 bounced sessions"], "count"],
    ],
    columns=["metric", "value", "value_type"],
)
summary["formatted_value"] = summary.apply(
    lambda row: f"{row['value']:.1%}" if row["value_type"] == "rate" else f"{int(row['value']):,}",
    axis=1,
)
summary[["metric", "formatted_value"]]


In [ ]:
digital_journey = pd.DataFrame(
    {
        "stage": [
            "Crisis supporter page cohort",
            "Reached digital support page",
            "Clicked Teams signup",
        ],
        "sessions": [
            cohort_sessions,
            int(sessions["reached_digital_after_crisis"].sum()),
            int(sessions["reached_digital_signup_after_digital"].sum()),
        ],
    }
)
digital_journey["share_of_cohort"] = digital_journey["sessions"] / cohort_sessions

digital_journey["label"] = digital_journey.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_cohort']:.1%} of cohort",
    axis=1,
)

phone_journey = pd.DataFrame(
    {
        "stage": [
            "Crisis supporter page cohort",
            "Reached phone support page",
            "Reached phone form page",
            "Submitted phone form",
        ],
        "sessions": [
            cohort_sessions,
            int(sessions["reached_phone_support_after_crisis"].sum()),
            int(sessions["reached_phone_form_after_support"].sum()),
            int(sessions["reached_phone_submit_after_form"].sum()),
        ],
    }
)
phone_journey["share_of_cohort"] = phone_journey["sessions"] / cohort_sessions
phone_journey["label"] = phone_journey.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_cohort']:.1%} of cohort",
    axis=1,
)


def make_ordered_bar(frame, title, color):
    fig = px.bar(
        frame,
        x="sessions",
        y="stage",
        orientation="h",
        text="label",
        template="lifeline",
        title=title,
    )
    fig.update_traces(marker_color=color, textposition="outside", cliponaxis=False)
    fig.update_layout(
        height=460 if len(frame) <= 3 else 520,
        margin=dict(l=240, r=140, t=90, b=60),
        xaxis_title="Sessions",
        yaxis_title="",
    )
    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(frame["stage"].tolist())))
    lifeline_theme.add_lifeline_logo(fig)
    return fig

make_ordered_bar(
    digital_journey,
    f"Ordered Digital Journey from Crisis Supporter Page ({analysis_window_label})",
    lifeline_theme.SUPPORT_BLUE,
).show()

make_ordered_bar(
    phone_journey,
    f"Ordered Phone Journey from Crisis Supporter Page ({analysis_window_label})",
    lifeline_theme.OPTIMISM_ORANGE,
).show()


In [ ]:
first_action_order = [
    "Digital page first",
    "Phone page first",
    "Other page first",
    "Other activity only",
    "No onward activity",
]
branch_choice = (
    sessions["first_post_crisis_action"]
    .value_counts()
    .reindex(first_action_order, fill_value=0)
    .rename_axis("first_action")
    .reset_index(name="sessions")
)
branch_choice["share_of_cohort"] = branch_choice["sessions"] / cohort_sessions
branch_choice["label"] = branch_choice.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_cohort']:.1%}",
    axis=1,
)
branch_choice

no_branch_mask = sessions["onward_outcome"] != "Reached tracked branch"
engaged_no_branch = int(((sessions["engaged_session"] == 1) & no_branch_mask).sum())
other_page_only = int((sessions["onward_outcome"] == "Other page only").sum())
other_activity_only = int((sessions["onward_outcome"] == "Other activity only").sum())
no_onward_activity = int((sessions["onward_outcome"] == "No onward activity").sum())

dropoff = pd.DataFrame(
    {
        "stage": [
            "No onward activity after crisis",
            "Other page only after crisis",
            "Other activity only after crisis",
            "Engaged but no tracked branch",
            "Reached digital page but no Teams signup",
            "Reached phone support but no phone form",
            "Reached phone form but no submit",
        ],
        "sessions": [
            no_onward_activity,
            other_page_only,
            other_activity_only,
            engaged_no_branch,
            int(sessions["reached_digital_after_crisis"].sum() - sessions["reached_digital_signup_after_digital"].sum()),
            int(sessions["reached_phone_support_after_crisis"].sum() - sessions["reached_phone_form_after_support"].sum()),
            int(sessions["reached_phone_form_after_support"].sum() - sessions["reached_phone_submit_after_form"].sum()),
        ],
    }
)
dropoff["share_of_cohort"] = dropoff["sessions"] / cohort_sessions
dropoff["label"] = dropoff.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_cohort']:.1%} of cohort",
    axis=1,
)

branch_fig = px.bar(
    branch_choice,
    x="sessions",
    y="first_action",
    orientation="h",
    text="label",
    template="lifeline",
    title=f"First Action After Crisis Supporter Page ({analysis_window_label})",
)
branch_fig.update_traces(marker_color=lifeline_theme.SUPPORT_BLUE, textposition="outside", cliponaxis=False)
branch_fig.update_layout(height=460, margin=dict(l=240, r=140, t=90, b=60), xaxis_title="Sessions", yaxis_title="")
branch_fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(first_action_order)))
lifeline_theme.add_lifeline_logo(branch_fig)
branch_fig.show()

dropoff_fig = px.bar(
    dropoff,
    x="sessions",
    y="stage",
    orientation="h",
    text="label",
    template="lifeline",
    title=f"Drop-off and Off-ramp Sessions from Crisis Cohort ({analysis_window_label})",
)
dropoff_fig.update_traces(marker_color=lifeline_theme.SUPPORT_BLUE, textposition="outside", cliponaxis=False)
dropoff_fig.update_layout(height=560, margin=dict(l=300, r=140, t=90, b=60), xaxis_title="Sessions", yaxis_title="")
dropoff_fig.update_yaxes(categoryorder="total ascending")
lifeline_theme.add_lifeline_logo(dropoff_fig)
dropoff_fig.show()


In [ ]:
digital_only = sessions[sessions["branch_mix"] == "Digital only"]
phone_only = sessions[sessions["branch_mix"] == "Phone only"]
both_branches = sessions[sessions["branch_mix"] == "Both branches"]
other_page_only_sessions = sessions[sessions["onward_outcome"] == "Other page only"]
other_activity_only_sessions = sessions[sessions["onward_outcome"] == "Other activity only"]
no_onward_sessions = sessions[sessions["onward_outcome"] == "No onward activity"]

nodes = [
    "Crisis supporter cohort",
    "Digital only",
    "Phone only",
    "Both branches",
    "Other page only",
    "Other activity only",
    "No onward activity",
    "Digital signup",
    "No digital signup",
    "Phone submit",
    "No phone submit",
    "Any conversion",
    "No conversion",
    "Engaged no tracked branch",
    "GA4 bounce / unengaged",
]
node_index = {name: idx for idx, name in enumerate(nodes)}

links = [
    ("Crisis supporter cohort", "Digital only", len(digital_only)),
    ("Crisis supporter cohort", "Phone only", len(phone_only)),
    ("Crisis supporter cohort", "Both branches", len(both_branches)),
    ("Crisis supporter cohort", "Other page only", len(other_page_only_sessions)),
    ("Crisis supporter cohort", "Other activity only", len(other_activity_only_sessions)),
    ("Crisis supporter cohort", "No onward activity", len(no_onward_sessions)),
    ("Digital only", "Digital signup", int(digital_only["reached_digital_signup_after_digital"].sum())),
    ("Digital only", "No digital signup", len(digital_only) - int(digital_only["reached_digital_signup_after_digital"].sum())),
    ("Phone only", "Phone submit", int(phone_only["reached_phone_submit_after_form"].sum())),
    ("Phone only", "No phone submit", len(phone_only) - int(phone_only["reached_phone_submit_after_form"].sum())),
    ("Both branches", "Any conversion", int(((both_branches["reached_digital_signup_after_digital"] == 1) | (both_branches["reached_phone_submit_after_form"] == 1)).sum())),
    ("Both branches", "No conversion", len(both_branches) - int(((both_branches["reached_digital_signup_after_digital"] == 1) | (both_branches["reached_phone_submit_after_form"] == 1)).sum())),
    ("Other page only", "Engaged no tracked branch", int((other_page_only_sessions["engaged_session"] == 1).sum())),
    ("Other page only", "GA4 bounce / unengaged", len(other_page_only_sessions) - int((other_page_only_sessions["engaged_session"] == 1).sum())),
    ("Other activity only", "Engaged no tracked branch", int((other_activity_only_sessions["engaged_session"] == 1).sum())),
    ("Other activity only", "GA4 bounce / unengaged", len(other_activity_only_sessions) - int((other_activity_only_sessions["engaged_session"] == 1).sum())),
    ("No onward activity", "Engaged no tracked branch", int((no_onward_sessions["engaged_session"] == 1).sum())),
    ("No onward activity", "GA4 bounce / unengaged", len(no_onward_sessions) - int((no_onward_sessions["engaged_session"] == 1).sum())),
]

sankey = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=nodes, pad=24, thickness=20),
            link=dict(
                source=[node_index[source] for source, target, value in links if value > 0],
                target=[node_index[target] for source, target, value in links if value > 0],
                value=[value for source, target, value in links if value > 0],
            ),
        )
    ]
)
sankey.update_layout(
    template="lifeline",
    title=f"GA4 session flow on /get-involved/volunteer/volunteer-as-a-crisis-supporter ({analysis_window_label})",
    height=620,
    margin=dict(l=40, r=40, t=90, b=40),
)
lifeline_theme.add_lifeline_logo(sankey)
sankey.show()


In [ ]:
daily = sessions.groupby("report_date", as_index=False).agg(
    crisis_cohort_sessions=("session_key", "nunique"),
    digital_signup_sessions=("reached_digital_signup_after_digital", "sum"),
    phone_submit_sessions=("reached_phone_submit_after_form", "sum"),
)

daily_long = daily.melt(
    id_vars="report_date",
    value_vars=["crisis_cohort_sessions", "digital_signup_sessions", "phone_submit_sessions"],
    var_name="metric",
    value_name="sessions",
)

daily_long["metric"] = daily_long["metric"].map(
    {
        "crisis_cohort_sessions": "Crisis supporter cohort sessions",
        "digital_signup_sessions": "Digital Teams signup sessions",
        "phone_submit_sessions": "Phone form submit sessions",
    }
)

trend_fig = px.line(
    daily_long,
    x="report_date",
    y="sessions",
    color="metric",
    template="lifeline",
    title=f"Crisis Supporter Journey Trend ({analysis_window_label})",
    labels={"report_date": "Date", "sessions": "Sessions", "metric": "Metric"},
)
trend_fig.update_layout(hovermode="x unified")
lifeline_theme.add_lifeline_logo(trend_fig)
trend_fig.show()

entry_stage = sessions["entry_stage"].fillna("Unclassified").value_counts().rename_axis("entry_stage").reset_index(name="sessions")
entry_stage["share_of_cohort"] = entry_stage["sessions"] / cohort_sessions
entry_stage
